In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

def generate_ultimate_trading_dataset():
    # 1. DEFINE TICKERS (Global Futures & FX - Core for Futures First)
    # ES=F (S&P 500), INR=X (USD/INR), GC=F (Gold), ZN=F (10Y Treasury Note)
    tickers = ['ES=F', 'INR=X', 'GC=F', 'ZN=F']

    all_market_data = []

    print("--- STEP 1: Ingesting Real-Time Market Data ---")
    for ticker in tickers:
        print(f"Downloading 1-minute interval data for: {ticker}...")
        t_obj = yf.Ticker(ticker)

        # 1-minute data is the gold standard for high-frequency analyst roles
        df = t_obj.history(period='8d', interval='1m')

        if df.empty:
            print(f"Warning: No data found for {ticker}")
            continue

        df.reset_index(inplace=True)
        df['Ticker'] = ticker

        # --- STEP 2: ANALYST FEATURE ENGINEERING (The 'Alpha') ---
        # Log Returns & Rolling Volatility (Standard Risk Metrics)
        df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
        df['Volatility_20m'] = df['Log_Return'].rolling(window=20).std()

        # Price Slippage / Gap Analysis
        df['Price_Gap'] = df['Open'] - df['Close'].shift(1)

        # Relative Strength Index (RSI) - Technical Indicator simulation
        delta = df['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df['RSI_14m'] = 100 - (100 / (1 + rs))

        # --- STEP 3: SIMULATING TRADING LOGS (The 'Budget vs Actuals' Logic) ---
        # We simulate 'Budget' as the Risk Limit and 'Actual' as the Trade Result
        np.random.seed(42) # For consistent data generation
        df['Risk_Limit_USD'] = 500.00  # Our 'Budget' per trade

        # Simulate trade entry/exit logic for the dashboard
        df['Entry_Price'] = df['Open']
        df['Exit_Price'] = df['Close']
        df['Trade_PnL'] = (df['Exit_Price'] - df['Entry_Price']) * 100 # Simulated multiplier

        # Calculate Slippage (Budgeted Entry vs Actual Entry)
        df['Slippage_USD'] = np.random.uniform(0.05, 0.50, len(df))

        # Max Drawdown Simulation (Cumulative PnL)
        df['Cum_PnL'] = df['Trade_PnL'].cumsum()
        df['Peak'] = df['Cum_PnL'].cummax()
        df['Drawdown'] = df['Cum_PnL'] - df['Peak']

        all_market_data.append(df)

    # --- STEP 4: FINAL INTEGRATION & EXPORT ---
    print("\n--- STEP 2: Finalizing Master Dataset ---")
    final_df = pd.concat(all_market_data)

    # Rename columns to match industry standards for SQL/Power BI
    final_df.rename(columns={'Datetime': 'Transaction_Time'}, inplace=True)

    # Export to CSV
    filename = 'Ultimate_FuturesFirst_Project_Data.csv'
    final_df.to_csv(filename, index=False)

    print(f"SUCCESS! {len(final_df)} rows generated.")
    print(f"File saved as: {filename}")

    # Quick Summary for validation
    print("\nSample Data Overview:")
    print(final_df[['Transaction_Time', 'Ticker', 'Close', 'Trade_PnL', 'Drawdown']].head())

if __name__ == "__main__":
    generate_ultimate_trading_dataset()

--- STEP 1: Ingesting Real-Time Market Data ---

--- STEP 2: Finalizing Master Dataset ---
SUCCESS! 32466 rows generated.
File saved as: Ultimate_FuturesFirst_Project_Data.csv

Sample Data Overview:
            Transaction_Time Ticker    Close  Trade_PnL  Drawdown
0  2026-04-06 00:00:00-04:00   ES=F  6613.75      150.0       0.0
1  2026-04-06 00:01:00-04:00   ES=F  6615.75      200.0       0.0
2  2026-04-06 00:02:00-04:00   ES=F  6615.50      -25.0     -25.0
3  2026-04-06 00:03:00-04:00   ES=F  6615.75       25.0       0.0
4  2026-04-06 00:04:00-04:00   ES=F  6617.50      125.0       0.0


In [ ]:
from google.colab import files
files.download('Ultimate_FuturesFirst_Project_Data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>